# **Truth probe, directional ablation and CCS: practice**

In the lesson “Refusal as a single direction”, the direction was computed as a difference of means, and the intervention removed it from the activations. In the lesson “Internal lie detectors and truthfulness”, the same tools read the truth of statements: a supervised probe after Marks and Tegmark and unsupervised CCS after Burns et al. There, everything was computed on synthetic activations. Here, the same three tools are applied to a real chat model, Qwen2.5-0.5B-Instruct: 24 layers, a residual stream of dimension 896.

In this practice, you will:

* build a set of simple statements “City X is in country Y”, true and false, and collect the model's activations on the last token of each statement at every layer;
* build a truth probe as a difference of means (mass-mean probe, [Marks and Tegmark, 2024](https://arxiv.org/abs/2310.06824)) and find from which layer it separates true statements from false ones;
* remove the found direction from the activations with directional ablation, as in [Arditi et al., 2024](https://arxiv.org/abs/2406.11717), and check whether truth stays linearly readable;
* train CCS ([Burns et al., 2023](https://arxiv.org/abs/2212.03827)) on “Yes” / “No” contrast pairs without labels and compare its accuracy with the supervised probe.

The tasks in the notebook are numbered the same way as in the lesson “Truth probe and CCS: Practice”.

## **Environment setup**
A CPU is enough: the model is small, and all computations here are forward passes and training of linear probes. The forward passes over all statements take about a minute.

In [ ]:
!pip install transformers==5.18.0 -q

In [ ]:
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)  # forward passes only; gradients are enabled only for CCS
torch.manual_seed(0)
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval()
print(model.config.num_hidden_layers, model.config.hidden_size)

## **Statements**

For each of 69 cities, we take a true statement “The city of X is in Y.” and a false one, where the country is replaced by a random other country from the same list. A third of the cities goes to the test set entirely, with both statements: this way the probe never sees a test city during training.

In [ ]:
pairs = [("Paris", "France"), ("Berlin", "Germany"), ("Madrid", "Spain"), ("Rome", "Italy"), ("Lisbon", "Portugal"),
         ("Vienna", "Austria"), ("Warsaw", "Poland"), ("Prague", "the Czech Republic"), ("Budapest", "Hungary"), ("Athens", "Greece"),
         ("Stockholm", "Sweden"), ("Oslo", "Norway"), ("Helsinki", "Finland"), ("Copenhagen", "Denmark"), ("Dublin", "Ireland"),
         ("London", "the United Kingdom"), ("Amsterdam", "the Netherlands"), ("Brussels", "Belgium"), ("Bern", "Switzerland"), ("Moscow", "Russia"),
         ("Kyiv", "Ukraine"), ("Ankara", "Turkey"), ("Cairo", "Egypt"), ("Nairobi", "Kenya"), ("Lagos", "Nigeria"),
         ("Accra", "Ghana"), ("Tokyo", "Japan"), ("Beijing", "China"), ("Seoul", "South Korea"), ("Bangkok", "Thailand"),
         ("Hanoi", "Vietnam"), ("Jakarta", "Indonesia"), ("Manila", "the Philippines"), ("New Delhi", "India"), ("Karachi", "Pakistan"),
         ("Tehran", "Iran"), ("Baghdad", "Iraq"), ("Riyadh", "Saudi Arabia"), ("Ottawa", "Canada"), ("Toronto", "Canada"),
         ("Chicago", "the United States"), ("Boston", "the United States"), ("Mexico City", "Mexico"), ("Havana", "Cuba"), ("Lima", "Peru"),
         ("Bogota", "Colombia"), ("Santiago", "Chile"), ("Buenos Aires", "Argentina"), ("Montevideo", "Uruguay"), ("Caracas", "Venezuela"),
         ("Sydney", "Australia"), ("Melbourne", "Australia"), ("Auckland", "New Zealand"), ("Reykjavik", "Iceland"), ("Tallinn", "Estonia"),
         ("Riga", "Latvia"), ("Vilnius", "Lithuania"), ("Bucharest", "Romania"), ("Sofia", "Bulgaria"), ("Belgrade", "Serbia"),
         ("Zagreb", "Croatia"), ("Casablanca", "Morocco"), ("Tunis", "Tunisia"), ("Addis Ababa", "Ethiopia"), ("Kampala", "Uganda"),
         ("Mumbai", "India"), ("Osaka", "Japan"), ("Shanghai", "China"), ("Istanbul", "Turkey")]

rng = np.random.default_rng(0)
countries = sorted({k for _, k in pairs})
statements, labels, city = [], [], []
for i, (c, k) in enumerate(pairs):
    wrong = rng.choice([x for x in countries if x != k])
    statements += [f"The city of {c} is in {k}.", f"The city of {c} is in {wrong}."]
    labels += [1, 0]
    city += [i, i]
y = torch.tensor(labels)
test_cities = set(rng.permutation(len(pairs))[: len(pairs) // 3].tolist())
test = torch.tensor([c in test_cities for c in city])
train = ~test
print(len(statements), "statements;", int(test.sum()), "test")
print(statements[:4])

## **Activations**

Run every statement through the model and store the residual stream state on the last token (the period) at every layer: `hidden_states[0]` is the embeddings, `hidden_states[l]` is the output of block $l$.

In [ ]:
def last_token_states(texts):
    out = []
    for s in texts:
        ids = tok(s, return_tensors="pt")
        hs = model(**ids, output_hidden_states=True).hidden_states
        out.append(torch.stack([h[0, -1] for h in hs]))
    return torch.stack(out)  # [N, 25, 896]

H = last_token_states(statements)
print(H.shape)

## **Difference-of-means probe by layer**

The difference-of-means probe is built as in the code of Marks and Tegmark ([`geometry-of-truth`](https://github.com/saprmarks/geometry-of-truth): the function `collect_acts` with `center=True` and the class `MMProbe` in `probes.py`). The activations are centered by the mean over all statements, the direction $r = \bar h_{\text{true}} - \bar h_{\text{false}}$ is computed over the training statements, and a statement is classified as true if the projection of its centered activation onto $r$ is positive: this is the threshold $\sigma(\theta_{mm}^\top h) > 0.5$ from the paper. Compute the test accuracy of this probe at every layer.

In [ ]:
def mass_mean(X):
    Xc = X - X.mean(0)  # centering over all statements, as collect_acts(center=True) in Marks and Tegmark
    r = ...  # Your code here
    pred = ...  # Your code here
    return r, (pred[test] == y[test]).float().mean().item()

acc = [mass_mean(H[:, l])[1] for l in range(H.shape[1])]
for l, a in enumerate(acc):
    print(l, round(a, 3))

**Task 1. From which layer on is the test accuracy of the difference-of-means probe at least 0.9? Write the layer number.**

**Task 2. What is the test accuracy of the difference-of-means probe at layer 16? Round to two decimal places.**

## **Directional ablation of the truth direction**

Take layer 16. Directional ablation removes from every activation its component along the unit direction $\hat r = r/\lVert r\rVert$:

$$h \leftarrow h - (\hat r^\top h)\,\hat r.$$

After that, a difference-of-means probe computed on the same activations is useless by construction: the difference of means after ablation is the zero vector. The question is a different one: does truth stay linearly readable along other directions? Check it with logistic regression (features standardized over the training statements) trained on the activations before and after ablation. The activations are taken in float64, so that the projection onto $\hat r$ after ablation is zero up to rounding.

In Arditi et al., ablation runs during the forward pass, at every layer and every token position, and changes the model's answer. Here it is applied to the stored activations of one layer: the question is not about the model's behavior but about what stays linearly readable at this layer.

In [ ]:
L = 16
X = H[:, L].double()
r, _ = mass_mean(X)
r_hat = r / r.norm()
X_abl = ...  # Your code here

def logistic_acc(Z):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000))
    clf.fit(Z[train].numpy(), y[train].numpy())
    return clf.score(Z[test].numpy(), y[test].numpy())

print("max |projection| after ablation:", (X_abl @ r_hat).abs().max().item())
print("logistic before:", round(logistic_acc(X), 3))
print("logistic after: ", round(logistic_acc(X_abl), 3))

**Task 3. What is the test accuracy of the logistic probe at layer 16 after directional ablation of the direction $\hat r$? Round to two decimal places.**

**Task 4. What does the result of task 3 show?**

## **CCS: truth without labels**

For every statement, CCS builds a contrast pair: “Is it true that the city of X is in Y? Yes” and the same question with the answer “No”. The activations of each half of the pair are centered separately, as in the function `normalize` of the authors' code ([`discovering_latent_knowledge`](https://github.com/collin-burns/discovering_latent_knowledge), `utils.py`): the mean over the training statements is subtracted from the training statements, and the mean over the test statements from the test statements. This way the probe cannot read the “Yes” or “No” token itself. The probe $p(h) = \sigma(w^\top h + b)$ is trained without labels on the loss

$$L = \big(p^+ - (1 - p^-)\big)^2 + \min(p^+, p^-)^2,$$

where the first term demands consistency ($p^+ \approx 1 - p^-$) and the second forbids the degenerate answer $p^+ = p^- = 0.5$. CCS does not fix the sign of the direction, so accuracy is taken as $\max(\text{acc}, 1 - \text{acc})$. As in the authors' code, train ten probes from different random initializations (AdamW, learning rate $10^{-3}$, weight decay 0.01, 1000 epochs on all training pairs) and keep the one with the lowest loss on the training pairs.

*The paper and the authors' code differ here. In the paper (Section 2.2 and Appendix G), after centering the activations are also divided by their average norm, and the learning rate is 0.01; in the code, by default the scale is not normalized (`var_normalize=False`), and the learning rate is $10^{-3}$. The notebook follows the code.*

In [ ]:
questions = [s[:-1].replace("The city of", "Is it true that the city of") + "?" for s in statements]
Hp = last_token_states([q + " Yes" for q in questions])[:, L]
Hn = last_token_states([q + " No" for q in questions])[:, L]

def normalize(A, split):  # each half of the pair and each split are centered by their own mean
    return A[split] - A[split].mean(0)

Ptr, Ntr, Pte, Nte = normalize(Hp, train), normalize(Hn, train), normalize(Hp, test), normalize(Hn, test)

def ccs_loss(w, b, P, N):
    pp, pn = torch.sigmoid(P @ w + b), torch.sigmoid(N @ w + b)
    return ...  # Your code here

best = None
with torch.enable_grad():
    for seed in range(10):
        torch.manual_seed(seed)
        lin = torch.nn.Linear(Hp.shape[1], 1)  # initialization of nn.Linear, as in the authors' code
        w, b = lin.weight[0], lin.bias
        opt = torch.optim.AdamW(lin.parameters(), lr=1e-3, weight_decay=0.01)
        for _ in range(1000):
            loss = ccs_loss(w, b, Ptr, Ntr)
            opt.zero_grad(); loss.backward(); opt.step()
        if best is None or loss.item() < best[0]:
            best = (loss.item(), w.detach(), b.detach(), seed)

loss, w, b, seed = best
score = 0.5 * (torch.sigmoid(Pte @ w + b) + 1 - torch.sigmoid(Nte @ w + b))
a = ((score > 0.5).long() == y[test]).float().mean().item()
print("seed", seed, "loss", round(loss, 4))
print("CCS accuracy:", round(max(a, 1 - a), 3))

**Task 5. What is the test accuracy of CCS at layer 16? Round to two decimal places.**

## **Conclusion**

- The truth of simple statements about cities in Qwen2.5-0.5B-Instruct is linearly readable: a difference-of-means probe trained on two thirds of the cities separates true statements from false ones on the rest, but only from layer 8 of 24 on: at the first four layers its accuracy is about 0.5, and at layers 4–7 it is from 0.61 to 0.67.
- Directional ablation of a single direction $\hat r$ is the same operation that removes the refusal direction in Arditi et al. When $\hat r$ is the difference of class means, the class means coincide after ablation, and no linear classifier separates the classes better than a constant: this is the condition for linear concept erasure from [Belrose et al., 2023](https://arxiv.org/abs/2306.03819) (LEACE). This is why the logistic probe drops to 0.5 after ablation, although before it was more accurate than the difference-of-means probe. A nonlinear probe, or the model itself at later layers, may still read truth: ablation at one layer erases only its linearly readable part.
- CCS finds a truth direction without labels: at layer 16 its accuracy is 0.957, the same as that of the difference-of-means probe and lower than that of the logistic probe (1.0); both probes are trained with labels. But a low CCS loss by itself does not prove that truth was found; this is the objection of Farquhar et al. (2023) from the lesson “Internal lie detectors and truthfulness”: here, the accuracy could be checked only because the statements have labels. Change `L` to 8 or 12 and see what happens to the CCS loss and accuracy.
- The set here is small: 46 test statements, so accuracy moves in steps of $1/46 \approx 0.022$. Compare accuracies with this in mind.